# Часть 1: Преобразование полученных данных в единый формат

In [1]:
import pandas as pd

## IMOEX

Используем разность логарифмов

In [3]:
imoex = pd.read_csv("../russian_data/MOEX Russia Index Historical Data.csv")

In [4]:
imoex = imoex[['Price', 'Date']]

In [5]:
imoex['Price'] = imoex['Price'].str.replace(',', '').astype(float)
imoex['Date'] = pd.to_datetime(imoex['Date'], format='%m/%d/%Y')

In [6]:
imoex = imoex.sort_values(by='Date')

In [7]:
imoex.rename(columns={"Date": "date", "Price": "IMOEX"}, inplace=True)

In [8]:
dates = imoex['date'].tolist()
for i in range(len(dates)-1):
    assert dates[i+1] == pd.date_range(start=dates[i], periods=2, freq="MS")[1]

In [9]:
assert pd.infer_freq(dates) == "MS"

In [12]:
imoex.to_csv("../russian_data/processed/imoex.csv", index=False)

## ИПЦ России

In [13]:
cpi = pd.read_csv("../russian_data/cpi_rus.csv")

In [14]:
cpi['date'] = pd.to_datetime(cpi['date'])

In [16]:
cpi.rename(columns={"value": "cpi_rus"}, inplace=True)

In [21]:
cpi['date'] = cpi['date']+ pd.Timedelta(days=1)

In [22]:
pd.infer_freq(cpi['date'])

'MS'

In [23]:
dates = cpi['date'].tolist()
for i in range(len(dates)-1):
    assert dates[i+1] == pd.date_range(start=dates[i], periods=2, freq="MS")[1]

In [24]:
cpi.to_csv("../russian_data/processed/cpi_rus.csv", index=False)

## Мировая добыча и российская

In [26]:
oil = pd.read_excel("../russian_data/oil_prod.xlsx")
df = oil.T
df = df.reset_index()
df.drop(labels=['index'], axis=1, inplace=True)

In [27]:
# Всё в одном блоке
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)
df.columns = ['date', 'world', 'russia']

# Конвертация всех типов
df['world'] = pd.to_numeric(df['world'], errors='coerce')
df['russia'] = pd.to_numeric(df['russia'], errors='coerce')
df['date'] = pd.to_datetime(df['date'], format='%b %Y').dt.to_period('M').dt.start_time


In [28]:
df.rename(columns={"world": "oil_prod_world", "russia": "oil_prod_russia"}, inplace=True)

In [29]:
df = df.reset_index()


In [30]:
dates = df['date'].tolist()
for i in range(len(dates)-1):
    assert dates[i+1] == pd.date_range(start=dates[i], periods=2, freq="MS")[1]

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 638 entries, 0 to 637
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   index            638 non-null    int64         
 1   date             638 non-null    datetime64[us]
 2   oil_prod_world   638 non-null    float64       
 3   oil_prod_russia  369 non-null    float64       
dtypes: datetime64[us](1), float64(2), int64(1)
memory usage: 20.1 KB


In [35]:
oil_russia = pd.read_excel("../russian_data/oil_russia.xlsx")
df = oil_russia.T
df = df.reset_index()
df.columns = df.iloc[0]
df = df[2:].reset_index(drop=True)
df.columns = ['date', 'russia']
df['date'] = pd.to_datetime(df['date'], format='%b %Y').dt.to_period('M').dt.start_time

import numpy as np

def timestamp_to_float(ts):
    """Преобразует Timestamp в формат ГГГГ.ММДД"""
    if isinstance(ts, pd.Timestamp):
        # Год + месяц/100 + день/10000
        return ts.year + ts.month / 100 + ts.day / 10000
    return ts


def clean_with_timestamp(x):
    if x == '--':
        return np.nan
    if isinstance(x, pd.Timestamp):
        return x.year + x.month / 100 + x.day / 10000
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan


cleaned_list = [clean_with_timestamp(x) for x in df['russia'].tolist()]

In [36]:
df['russia'] = cleaned_list
df = df[~df['russia'].isna()]
pd.infer_freq(df['date'])
dates = df['date'].tolist()
for i in range(len(dates)-1):
    assert dates[i+1] == pd.date_range(start=dates[i], periods=2, freq="MS")[1]

df.rename(columns={"russia": "oil_prod_russia"}, inplace=True)

In [38]:
df.to_csv("../russian_data/processed/oil_prod_russia.csv", index=False)